# 第308章: GANSpace — 教師なし属性発見

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] GANSpaceの核心アイデア（PCAで主要変動方向を発見）を説明できる
- [ ] 潜在空間のPCA主成分を計算し、各成分が制御する属性を分析できる
- [ ] 教師あり（InterFaceGAN）と教師なし（GANSpace）の違いを比較できる
- [ ] レイヤー別の操作が画像の異なるレベルに影響することを理解できる

## 🎯 前提知識

- ✅ Notebook 307（InterFaceGAN、属性方向の発見）
- ✅ Notebook 301（PCA分析）
- ✅ PCAの基本概念（主成分、固有ベクトル）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★☆☆（中級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに

InterFaceGAN（307章）は強力ですが、**属性ラベルが必要**でした。
**GANSpace**は、ラベルなしで属性方向を発見する教師なし手法です。

### 🤔 GANSpaceの核心アイデア

> **潜在空間でのデータの主な変動方向（PCA主成分）は、
> 意味的な属性の変化方向に対応している。**

```
  InterFaceGAN:                    GANSpace:
  ────────────                     ────────

  必要: 属性ラベル                   必要: なにもいらない！
        ↓                                 ↓
  SVM で境界面を学習                  PCA で主成分を計算
        ↓                                 ↓
  法線ベクトル = 属性方向              主成分ベクトル = 変動方向
                                          ↓
                                    各主成分を走査して意味を発見
```

### 📝 この章の構成

1. **GANSpaceの原理** — PCA主成分＝意味的変動方向
2. **PCA主成分の計算と走査** — 各主成分の意味を発見
3. **InterFaceGANとの比較** — 教師あり vs 教師なし
4. **レイヤー別操作の模倣** — 粗い変化 vs 細かい変化
5. **教師なしの限界と展望** — いつGANSpaceを使うべきか

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Device: {device}")
print("✅ 環境設定完了")

In [ ]:
# ============================================================
# VAEの学習（307章と同じ構成）
# ============================================================

class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
        return self.decode(z), mu, logvar

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model = VAE(latent_dim=20).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("VAE (z=20) 学習中...")
for epoch in range(20):
    model.train()
    total = 0
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device)
        recon, mu, logvar = model(x)
        loss = nn.functional.binary_cross_entropy(recon, x, reduction='sum') \
               - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20 | Loss: {total/len(train_ds):.2f}")

model.eval()
all_z, all_labels, all_images = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        mu, _ = model.encode(x.view(-1, 784).to(device))
        all_z.append(mu.cpu().numpy())
        all_labels.append(y.numpy())
        all_images.append(x.numpy())

all_z = np.concatenate(all_z)
all_labels = np.concatenate(all_labels)
all_images = np.concatenate(all_images)
digit_indices = {d: np.where(all_labels == d)[0] for d in range(10)}

print("✅ VAE学習・エンコード完了")

---

## 1. GANSpaceの原理: PCA主成分 = 意味的変動方向

GANSpace の手順はシンプルです：

1. 大量の潜在ベクトル z を収集
2. PCA を適用して主成分ベクトルを求める
3. 各主成分方向に走査して、何が変わるか観察

### なぜPCA主成分が「意味的」な方向になるのか？

潜在空間のデータ分布の分散が最大の方向は、
**生成画像の変動が最も大きい方向**に対応します。
画像の主要な変動は通常、意味的な属性（太さ、傾き、形状等）と相関します。

In [ ]:
# ============================================================
# GANSpace: PCA主成分の計算
# ============================================================

# 潜在空間全体でPCAを計算
pca = PCA(n_components=20)
z_pca = pca.fit_transform(all_z)

# 寄与率の表示
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(20), pca.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('主成分', fontsize=12)
axes[0].set_ylabel('寄与率', fontsize=12)
axes[0].set_title('PCA寄与率', fontsize=14, fontweight='bold')

cumulative = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(20), cumulative, 'o-', color='coral', linewidth=2)
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('主成分数', fontsize=12)
axes[1].set_ylabel('累積寄与率', fontsize=12)
axes[1].set_title('累積寄与率', fontsize=14, fontweight='bold')

n_90 = np.argmax(cumulative >= 0.9) + 1
axes[1].axvline(x=n_90 - 1, color='green', linestyle='--', alpha=0.5)
axes[1].text(n_90, 0.85, f'90%到達: {n_90}主成分', fontsize=11, color='green')

plt.tight_layout()
plt.savefig('fig_308_01_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"💡 上位 {n_90} 主成分で全分散の90%を説明")
print(f"   第1主成分の寄与率: {pca.explained_variance_ratio_[0]:.3f}")
print(f"   → 第1主成分が最も「意味的に重要」な変動方向")

In [ ]:
# ============================================================
# GANSpace: 上位主成分の走査マップ
# ============================================================

def decode_z(z_vec):
    with torch.no_grad():
        z_t = torch.tensor(z_vec, dtype=torch.float32).unsqueeze(0).to(device)
        return model.decode(z_t).cpu().numpy().reshape(28, 28)

n_components_show = 8
n_steps = 9
alphas = np.linspace(-3, 3, n_steps)

# 数字3で走査
base_idx = digit_indices[3][0]
z_base = all_z[base_idx]

fig, axes = plt.subplots(n_components_show, n_steps, figsize=(14, n_components_show * 1.5))

for comp in range(n_components_show):
    direction = pca.components_[comp]  # PCA主成分ベクトル

    for col, alpha in enumerate(alphas):
        z_mod = z_base + alpha * direction
        img = decode_z(z_mod)
        axes[comp, col].imshow(img, cmap='gray')
        axes[comp, col].axis('off')
        if col == 0:
            var_pct = pca.explained_variance_ratio_[comp] * 100
            axes[comp, col].set_ylabel(f'PC{comp+1}\n{var_pct:.1f}%',
                                       fontsize=9, rotation=0, labelpad=35)
        if comp == 0:
            axes[comp, col].set_title(f'α={alpha:.1f}', fontsize=9)

fig.suptitle('GANSpace: PCA主成分方向の走査マップ（数字3）\n各行 = 1つの主成分, α = -3 → +3',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_308_02_ganspace_traversal.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 各主成分が何を制御しているか観察:")
print("   PC1: 最も変動が大きい属性（数字の種類？形状？）")
print("   PC2-4: 次に重要な属性（太さ？傾き？）")
print("   PC5以降: 細かな変動")

In [ ]:
# ============================================================
# 異なる数字での主成分走査比較
# ============================================================

fig, axes = plt.subplots(5, n_steps, figsize=(14, 8))
digits_show = [0, 1, 3, 7, 9]

# 最も「面白い」主成分（第1主成分）で全数字を走査
comp = 0
direction = pca.components_[comp]

for row, digit in enumerate(digits_show):
    idx = digit_indices[digit][0]
    z_d = all_z[idx]

    for col, alpha in enumerate(alphas):
        z_mod = z_d + alpha * direction
        img = decode_z(z_mod)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(f'{digit}', fontsize=14, rotation=0, labelpad=20)
        if row == 0:
            axes[row, col].set_title(f'α={alpha:.1f}', fontsize=9)

fig.suptitle(f'PC1方向（寄与率{pca.explained_variance_ratio_[0]*100:.1f}%）の走査 — 異なる数字\n同じ主成分は各数字に一貫した変化を与えるか？',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_308_03_cross_digit_pc.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 2. InterFaceGAN vs GANSpace: 教師あり vs 教師なし

### 比較表

| 特性 | InterFaceGAN (307章) | GANSpace (本章) |
|------|---------------------|----------------|
| ラベル | 必要（教師あり） | 不要（教師なし） |
| 方向の発見方法 | SVM境界面の法線 | PCA主成分 |
| 方向の意味 | 指定した属性に対応 | 自動発見（解釈が必要） |
| 精度 | 指定属性に対して高い | 属性と主成分が一致するとは限らない |
| 汎用性 | ラベルがある属性のみ | 任意のデータに適用可能 |

In [ ]:
# ============================================================
# InterFaceGAN vs GANSpace の定量比較
# PCA主成分とSVM方向のコサイン類似度
# ============================================================

# 属性ラベルの再計算（307章と同じ）
def compute_attributes(images):
    n = len(images)
    imgs = images.reshape(n, 28, 28)
    attrs = {}

    ink = imgs.sum(axis=(1, 2))
    attrs['thickness'] = (ink > np.median(ink)).astype(int)

    slants = []
    for img in imgs:
        ys, xs = np.where(img > 0.3)
        if len(xs) < 10: slants.append(0); continue
        upper, lower = ys < 14, ys >= 14
        if upper.sum() < 3 or lower.sum() < 3: slants.append(0)
        else:
            slants.append(np.average(xs[upper], weights=img[ys[upper], xs[upper]])
                         - np.average(xs[lower], weights=img[ys[lower], xs[lower]]))
    attrs['slant'] = (np.array(slants) > np.median(slants)).astype(int)

    heights = []
    for img in imgs:
        ys, xs = np.where(img > 0.3)
        heights.append(np.average(ys, weights=img[ys, xs]) if len(ys) > 0 else 14)
    attrs['height'] = (np.array(heights) > np.median(heights)).astype(int)

    return attrs

attributes = compute_attributes(all_images)

# SVM方向の計算（InterFaceGAN）
svm_directions = {}
for attr_name, attr_labels in attributes.items():
    scaler = StandardScaler()
    z_s = scaler.fit_transform(all_z)
    svm = LinearSVC(C=1.0, max_iter=5000, random_state=42)
    svm.fit(z_s, attr_labels)
    d = svm.coef_[0] / scaler.scale_
    svm_directions[attr_name] = d / np.linalg.norm(d)

# PCA方向 vs SVM方向の類似度
attr_names = list(attributes.keys())
attr_labels_jp = {'thickness': '太さ', 'slant': '傾き', 'height': '上下位置'}

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

n_pc = 10
similarity_matrix = np.zeros((len(attr_names), n_pc))

for i, attr in enumerate(attr_names):
    for j in range(n_pc):
        similarity_matrix[i, j] = abs(np.dot(svm_directions[attr], pca.components_[j]))

im = ax.imshow(similarity_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_pc))
ax.set_xticklabels([f'PC{j+1}' for j in range(n_pc)])
ax.set_yticks(range(len(attr_names)))
ax.set_yticklabels([attr_labels_jp[a] for a in attr_names], fontsize=12)

for i in range(len(attr_names)):
    for j in range(n_pc):
        ax.text(j, i, f'{similarity_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
    best_pc = np.argmax(similarity_matrix[i])
    ax.text(best_pc, i, f'{similarity_matrix[i,best_pc]:.2f}', ha='center', va='center',
           fontsize=10, fontweight='bold', color='red')

plt.colorbar(im, label='|コサイン類似度|')
ax.set_title('SVM属性方向 vs PCA主成分の類似度\n赤太字 = 最も一致する主成分',
             fontsize=14, fontweight='bold')
ax.set_xlabel('PCA主成分', fontsize=12)
plt.tight_layout()
plt.savefig('fig_308_04_svm_vs_pca.png', dpi=150, bbox_inches='tight')
plt.show()

for attr in attr_names:
    best_pc = np.argmax([abs(np.dot(svm_directions[attr], pca.components_[j])) for j in range(n_pc)])
    best_sim = abs(np.dot(svm_directions[attr], pca.components_[best_pc]))
    print(f"  {attr_labels_jp[attr]}: 最も近い主成分 = PC{best_pc+1} (類似度 {best_sim:.3f})")

---

## 3. レイヤー別操作のシミュレーション

StyleGANの本来のGANSpaceでは、**異なるレイヤー**で操作することで
変化の「解像度」を制御できます。

```
  W空間の操作レイヤー        影響範囲
  ──────────────            ────────
  初期レイヤー (4×4→8×8)     全体構造（姿勢、形状）
  中間レイヤー (16×16→32×32) 中間特徴（表情、太さ）
  後半レイヤー (64×64→256)   細部（テクスチャ、色）
```

MNISTのシンプルなVAEではレイヤー別操作はできませんが、
**デコーダの異なる層で操作する**ことで擬似的に模倣できます。

In [ ]:
# ============================================================
# デコーダのレイヤー別操作のシミュレーション
# 中間特徴量に主成分方向を加える
# ============================================================

class VAEWithIntermediateAccess(nn.Module):
    """中間層にアクセス可能なVAE"""
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # デコーダを段階的に定義
        self.dec_layer1 = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU())
        self.dec_layer2 = nn.Sequential(nn.Linear(128, 256), nn.ReLU())
        self.dec_layer3 = nn.Sequential(nn.Linear(256, 784), nn.Sigmoid())

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode_with_perturbation(self, z, layer_idx=None, perturbation=None):
        """指定レイヤーの出力に摂動を加えてデコード"""
        h = self.dec_layer1(z)
        if layer_idx == 1 and perturbation is not None:
            h = h + perturbation[:, :h.shape[1]]
        h = self.dec_layer2(h)
        if layer_idx == 2 and perturbation is not None:
            h = h + perturbation[:, :h.shape[1]]
        h = self.dec_layer3(h)
        return h

    def decode(self, z):
        return self.dec_layer3(self.dec_layer2(self.dec_layer1(z)))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
        return self.decode(z), mu, logvar

# 元のモデルの重みをコピー
model_inter = VAEWithIntermediateAccess(latent_dim=20).to(device)
# 手動で重みをコピー
with torch.no_grad():
    model_inter.encoder.load_state_dict(model.encoder.state_dict())
    model_inter.fc_mu.load_state_dict(model.fc_mu.state_dict())
    model_inter.fc_logvar.load_state_dict(model.fc_logvar.state_dict())
    # デコーダレイヤー
    model_inter.dec_layer1[0].weight.copy_(model.decoder[0].weight)
    model_inter.dec_layer1[0].bias.copy_(model.decoder[0].bias)
    model_inter.dec_layer2[0].weight.copy_(model.decoder[2].weight)
    model_inter.dec_layer2[0].bias.copy_(model.decoder[2].bias)
    model_inter.dec_layer3[0].weight.copy_(model.decoder[4].weight)
    model_inter.dec_layer3[0].bias.copy_(model.decoder[4].bias)

model_inter.eval()

# 各レイヤーの中間出力からPCA主成分を計算
layer_outputs = {1: [], 2: []}
with torch.no_grad():
    for x, _ in test_loader:
        z = model_inter.encode(x.view(-1, 784).to(device))[0]
        h1 = model_inter.dec_layer1(z)
        h2 = model_inter.dec_layer2(h1)
        layer_outputs[1].append(h1.cpu().numpy())
        layer_outputs[2].append(h2.cpu().numpy())

for l in layer_outputs:
    layer_outputs[l] = np.concatenate(layer_outputs[l])

print("✅ 中間層出力の抽出完了")
for l, out in layer_outputs.items():
    print(f"  Layer {l}: shape = {out.shape}")

In [ ]:
# ============================================================
# レイヤー別PCA走査の比較
# ============================================================

# 各レイヤーでPCA
layer_pcas = {}
for l in [1, 2]:
    layer_pcas[l] = PCA(n_components=5)
    layer_pcas[l].fit(layer_outputs[l])

# 走査の比較: 潜在空間 vs Layer1 vs Layer2
fig, axes = plt.subplots(3, n_steps, figsize=(14, 5))

base_idx = digit_indices[3][0]
z_base_t = torch.tensor(all_z[base_idx], dtype=torch.float32).unsqueeze(0).to(device)

# Row 0: 潜在空間のPC1走査（標準的なGANSpace）
for col, alpha in enumerate(alphas):
    z_mod = all_z[base_idx] + alpha * pca.components_[0]
    img = decode_z(z_mod)
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].axis('off')
    if col == 0: axes[0, col].set_ylabel('潜在空間\nPC1', fontsize=10, rotation=0, labelpad=45)
    axes[0, col].set_title(f'α={alpha:.1f}', fontsize=9)

# Row 1: Layer1のPC1操作
for col, alpha in enumerate(alphas):
    pc_dir = torch.tensor(layer_pcas[1].components_[0], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        img = model_inter.decode_with_perturbation(z_base_t, layer_idx=1,
              perturbation=alpha * pc_dir).cpu().numpy().reshape(28, 28)
    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].axis('off')
    if col == 0: axes[1, col].set_ylabel('Layer1\nPC1', fontsize=10, rotation=0, labelpad=45)

# Row 2: Layer2のPC1操作
for col, alpha in enumerate(alphas):
    pc_dir = torch.tensor(layer_pcas[2].components_[0], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        img = model_inter.decode_with_perturbation(z_base_t, layer_idx=2,
              perturbation=alpha * pc_dir).cpu().numpy().reshape(28, 28)
    axes[2, col].imshow(img, cmap='gray')
    axes[2, col].axis('off')
    if col == 0: axes[2, col].set_ylabel('Layer2\nPC1', fontsize=10, rotation=0, labelpad=45)

fig.suptitle('レイヤー別操作の比較\n操作する層によって変化の「解像度」が異なる',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_308_05_layerwise_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 潜在空間操作: 全体構造の大きな変化")
print("   Layer1操作: 中間レベルの変化")
print("   Layer2操作: 出力に近い細かな変化")
print("   → StyleGANでは、この原理がW空間のレイヤー別操作に活用されている")

---

## 4. 教師なし手法の限界と使い分け

### いつGANSpaceを使うべきか？

| 状況 | 推奨手法 |
|------|---------|
| 属性ラベルがある | InterFaceGAN（精度が高い） |
| ラベルがない | GANSpace（唯一の選択肢） |
| 探索的分析（何があるか知りたい） | GANSpace（発見的） |
| 特定属性の精密制御 | InterFaceGAN + 直交化 |
| 新しいモデルの理解 | GANSpace（まず全体像を把握） |

### GANSpaceの限界

1. **PCA主成分 ≠ 意味的属性**: 主成分は分散最大の方向であり、人間が理解しやすい属性と一致するとは限らない
2. **線形制約**: PCAは線形変換なので、非線形な属性の変化は捉えにくい
3. **解釈が必要**: 各主成分の意味は手動で確認する必要がある

---

## まとめ

### 🎯 このノートブックで学んだこと

**GANSpaceの原理**
- ✓ PCA主成分 = 潜在空間での最大変動方向 ≈ 意味的属性方向
- ✓ ラベル不要で属性方向を発見可能（教師なし）

**InterFaceGANとの比較**
- ✓ InterFaceGAN: 精度高いがラベル必要
- ✓ GANSpace: ラベル不要だが主成分の解釈が必要
- ✓ PCA主成分とSVM方向の類似度で両者の関係を定量化

**レイヤー別操作**
- ✓ 異なるデコーダ層のPCA主成分は異なるレベルの変化を制御
- ✓ StyleGANのW空間レイヤー別操作の原理を理解

### ⚠️ よくある間違い

❌ 「GANSpaceはInterFaceGANの完全な代替」
✅ 特定の属性を精密に制御したい場合はInterFaceGANが優れる。GANSpaceは「何があるかの発見」に適する

❌ 「PCA主成分数 = 意味的属性の数」
✅ PCA主成分は数学的に定義される方向であり、人間が認識する属性と1対1対応するとは限らない

### ✅ 学習チェックリスト

- [ ] GANSpaceの手順を説明できるか？
- [ ] PCA主成分が意味的方向に対応する理由を説明できるか？
- [ ] InterFaceGANとGANSpaceの使い分けを説明できるか？

---

**次のステップ**: ノートブック309「エージング・シミュレーション」で、学んだ属性編集技術を応用して、数字画像の「老化・劣化」シミュレーションを実装します！

---

## 🎓 自己評価クイズ

### Q1: GANSpaceでPCA主成分を潜在空間に適用すると、なぜ意味のある属性変化が起きるのか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 潜在空間のデータの分散が最大の方向は、生成画像の変動が最も大きい属性に対応するため

well-trainedなGeneratorでは、潜在空間の構造が画像の意味的な構造を反映している。分散最大の方向は最も支配的な属性変化の方向に自然と対応する。

</details>

---

### Q2: GANSpaceとInterFaceGANで見つかる方向ベクトルの類似度が低い場合、何が考えられるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: その属性がデータの主要な変動方向ではない（寄与率が小さい）可能性が高い

PCA主成分は「分散が大きい方向」から順に並ぶため、分散が小さい属性の方向は下位の主成分にのみ含まれ、上位主成分とは一致しない。

</details>

---

### Q3: レイヤー別操作で、初期レイヤーと後半レイヤーの操作が異なる効果を持つのはなぜか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 初期レイヤーは全体構造を決定し、後半レイヤーは細部を調整する階層的な構造を持つため

ニューラルネットワークは階層的な特徴表現を学習する。初期層は低周波成分（大まかな形状）、後半層は高周波成分（テクスチャ、エッジ）に対応する。

</details>